In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script for generating Sales KPI metrics using reusable classes and functions in Databricks
# Purpose: Refactor sales KPI pipeline into classes and functions for maintainability and code reuse
# Author: Giang Nguyen
# Date: 2025-10-13
# Description: This script loads product, sales, and market share data from Unity Catalog tables, processes and joins them, calculates KPIs (YoY growth, market penetration flag, sales rank), and writes the results to the sales_kpi table. All logic is encapsulated in classes and functions for future extensibility.

# from pyspark.sql import SparkSession  # SparkSession is already available in Databricks
from pyspark.sql.functions import col, lit, when, sum as _sum, avg, round, year, expr, row_number, lag
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, StringType, DoubleType, LongType, DateType

class SalesKPIDataLoader:
    """
    Class to load and validate input data from Unity Catalog tables.

    Args:
        spark (SparkSession): Spark session object.

    Methods:
        load_table(table_name: str) -> DataFrame:
            Loads a table from Unity Catalog.
    """
    def __init__(self, spark):
        self.spark = spark

    def load_table(self, table_name: str):
        """
        Loads a table from Unity Catalog.

        Args:
            table_name (str): Fully qualified table name.

        Returns:
            DataFrame: Loaded DataFrame.
        """
        return self.spark.read.table(table_name)

class SalesKPIPipeline:
    """
    Class encapsulating the sales KPI pipeline logic.

    Args:
        spark (SparkSession): Spark session object.

    Methods:
        run_pipeline() -> DataFrame:
            Executes the pipeline and returns the final KPI DataFrame.
        write_output(df, table_name: str):
            Writes the final DataFrame to a managed table in overwrite mode.
    """
    def __init__(self, spark):
        self.spark = spark
        self.loader = SalesKPIDataLoader(spark)

    def load_and_prepare_data(self):
        """
        Loads and prepares product, sales, and market share data.

        Returns:
            Tuple[DataFrame, DataFrame, DataFrame]: product_df, sales_df, market_share_df
        """
        product_df = self.loader.load_table("purgo_playground.product_data")
        sales_df = self.loader.load_table("purgo_playground.product_sales_data")
        market_share_df = self.loader.load_table("purgo_playground.product_marketshare_data")
        return product_df, sales_df, market_share_df

    def clean_and_transform_sales(self, sales_df):
        """
        Cleans and transforms sales data.

        Args:
            sales_df (DataFrame): Raw sales DataFrame.

        Returns:
            DataFrame: Cleaned sales DataFrame.
        """
        # Filter out rows with null sales_amount and add sales_year
        sales_df = sales_df.filter(col("sales_amount").isNotNull()) \
                           .withColumn("sales_year", year(col("sales_date")))
        return sales_df

    def join_product_info(self, sales_df, product_df):
        """
        Joins sales data with product info.

        Args:
            sales_df (DataFrame): Sales DataFrame.
            product_df (DataFrame): Product DataFrame.

        Returns:
            DataFrame: Enriched sales DataFrame.
        """
        # Rename product_id in product_df to avoid duplicate column after join
        product_df_renamed = product_df.withColumnRenamed("product_id", "prod_product_id")
        sales_enriched_df = sales_df.join(product_df_renamed, sales_df.sales_product_id == product_df_renamed.prod_product_id, "left") \
                                    .drop("prod_product_id")
        return sales_enriched_df

    def join_market_share_info(self, sales_enriched_df, market_share_df):
        """
        Joins sales data with market share info.

        Args:
            sales_enriched_df (DataFrame): Sales + product DataFrame.
            market_share_df (DataFrame): Market share DataFrame.

        Returns:
            DataFrame: Sales DataFrame with market share info.
        """
        # Rename ms_product_id in market_share_df to avoid duplicate column after join
        market_share_df_renamed = market_share_df.withColumnRenamed("ms_product_id", "ms_product_id")
        sales_market_df = sales_enriched_df.join(market_share_df_renamed, sales_enriched_df.sales_product_id == market_share_df_renamed.ms_product_id, "left") \
                                           .drop("ms_product_id")
        return sales_market_df

    def calculate_kpis(self, sales_market_df):
        """
        Calculates KPI metrics: total sales, YoY growth, market penetration flag, sales rank.

        Args:
            sales_market_df (DataFrame): Sales DataFrame with product and market share info.

        Returns:
            DataFrame: DataFrame with KPI metrics.
        """
        window_spec = Window.partitionBy("sales_product_id").orderBy("sales_year")
        # Aggregate total sales and average market share
        sales_agg_df = sales_market_df.groupBy("sales_product_id", "sales_year", "product_name", "market_segment") \
            .agg(
                _sum("sales_amount").alias("total_sales"),
                round(avg("market_share_pct"), 2).alias("avg_market_share")
            )
        # Previous year sales
        sales_agg_df = sales_agg_df.withColumn("prev_year_sales", lag("total_sales", 1).over(window_spec))
        # YoY growth percentage
        sales_agg_df = sales_agg_df.withColumn("yoy_growth_pct",
            round(((col("total_sales") - col("prev_year_sales")) / col("prev_year_sales")) * 100, 2)
        )
        # Market penetration flag
        sales_agg_df = sales_agg_df.withColumn("market_penetration_flag",
            when(col("avg_market_share") > 25, lit("High"))
             .when((col("avg_market_share") <= 25) & (col("avg_market_share") >= 10), lit("Medium"))
             .otherwise(lit("Low"))
        )
        # Sales rank per year
        rank_window = Window.partitionBy("sales_year").orderBy(col("total_sales").desc())
        sales_agg_df = sales_agg_df.withColumn("sales_rank", row_number().over(rank_window))
        return sales_agg_df

    def select_final_columns(self, sales_agg_df):
        """
        Selects and orders final output columns.

        Args:
            sales_agg_df (DataFrame): DataFrame with KPI metrics.

        Returns:
            DataFrame: Final output DataFrame.
        """
        final_df = sales_agg_df.select(
            col("sales_year").cast(IntegerType()),
            col("sales_product_id").cast(StringType()),
            col("product_name").cast(StringType()),
            col("market_segment").cast(StringType()),
            col("total_sales").cast(LongType()),
            col("prev_year_sales").cast(LongType()),
            col("yoy_growth_pct").cast(DoubleType()),
            col("avg_market_share").cast(DoubleType()),
            col("market_penetration_flag").cast(StringType()),
            col("sales_rank").cast(IntegerType())
        )
        return final_df

    def run_pipeline(self):
        """
        Runs the full sales KPI pipeline.

        Returns:
            DataFrame: Final KPI DataFrame.
        """
        product_df, sales_df, market_share_df = self.load_and_prepare_data()
        sales_df_clean = self.clean_and_transform_sales(sales_df)
        sales_enriched_df = self.join_product_info(sales_df_clean, product_df)
        sales_market_df = self.join_market_share_info(sales_enriched_df, market_share_df)
        sales_agg_df = self.calculate_kpis(sales_market_df)
        final_df = self.select_final_columns(sales_agg_df)
        return final_df

    def write_output(self, df, table_name: str):
        """
        Writes the final DataFrame to a managed table in overwrite mode.

        Args:
            df (DataFrame): DataFrame to write.
            table_name (str): Fully qualified output table name.

        Returns:
            None
        """
        df.write.mode("overwrite").saveAsTable(table_name)

# --- Main pipeline execution ---
def main(spark):
    """
    Main function to execute the Sales KPI pipeline.

    Args:
        spark (SparkSession): Spark session object.

    Returns:
        None
    """
    pipeline = SalesKPIPipeline(spark)
    final_df = pipeline.run_pipeline()
    # Display the DataFrame for interactive review
    display(final_df)
    # Write output to managed table
    pipeline.write_output(final_df, "purgo_playground.sales_kpi")

# Execute main pipeline
main(spark)
# End of script
